# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHAAkash/-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste the token itself
# into a cell -- this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_march":  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    "fact_daily_april":  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Confirms which table(s) claim above (touches Parquet metadata only, not data -- fast even at 79M rows)
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:18} {n:>12,} rows")


dim_clients                 104 rows
dim_content             519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily           78,835,655 rows
fact_daily_march      9,841,378 rows
fact_daily_april     10,424,730 rows
fact_query_90d        2,414,248 rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**4. What I'd predict (label / proxy):** `is_declining` = did this content item's April
impressions drop by more than 20% versus March? (`imp_apr < 0.8 * imp_mar`). This mirrors the
starter dataset's proxy from ML-02/ML-03, but built here from a genuine future window — March
features, April outcome — instead of a same-window trend bucket. That was the exact gap I
flagged in `w02_ml_task_framing.ipynb` as something to fix before the real capstone claim; this
notebook is that fix.

**5. One thing I deliberately exclude:** `fact_content_query_90d`'s columns
(`content_visible_query_count`, `rare_impressions_share`, `anonymized_impressions_share`,
`impressions_90d`). The skill notes this table's 90-day window overlaps the panel's *final*
months and isn't re-cut per decision point — for a March 2026 decision point, that window likely
extends past April, meaning it could carry information from *after* the label window. I'm not
using it as a feature until I've verified the window's actual start/end date against my decision
point. See Section 4 for the check.

**Field buckets, in full:**

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (all from the **prev30**, March 2–31 window only) | Fully observed before the decision point (2026-03-31); safe. |
| **Label / proxy** | `gsc_impressions` summed over **April 1–30** (`imp_apr`), and `is_declining` computed from it | The thing being predicted, or directly computed from it — never also a feature. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | For grouping, joining, and a client-grouped split — never fed to a model as a signal. |
| **Excluded** | `fact_content_query_90d.*` (window-alignment unverified for this decision point); any `ga4_*` column on rows where `ga4_data_available` is not `TRUE` (zero-filled, not real zeros — see `flyrank-data` skill); raw client/content identifiers as anything other than context | Each risks leakage, misalignment, or is simply not what it appears to be (a filled zero, not an observed zero). |


In [27]:
# Supports the bucket table above: confirm the actual columns on the two tables I'm
# choosing NOT to use as features yet, so the exclusion is based on what's really there.
print("fact_daily columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']} LIMIT 1").df()[["column_name", "column_type"]])
print()
print("fact_query_90d columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']} LIMIT 1").df()[["column_name", "column_type"]])


fact_daily columns:
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIG

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a
guess.* This section holds the three required verification queries, the five-feature frame, and
the deliberate-leak experiment, in that order.

### 3a. Three verification queries, on `month=2026-03`


In [28]:
# Query 1 -- GRAIN: one row really is one report_date x client x content, in March.
# Zero rows back means the grain holds.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"duplicate-grain rows found: {len(grain_check)} (expect 0)")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0 (expect 0)


,report_date,client_hash_id,content_hash_id,c


In [29]:
# Query 2 -- SLICE SIZE + DATE SPAN: row count and date span for the March partition.
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily_march']}
""").df()
span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [30]:
# Query 3 -- AVAILABILITY, checked with IS TRUE: how many March rows have real GA4 data
# (vs. zero-filled rows before a client's ga4_data_start)?
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily_march']}
""").df()
avail["pct_available"] = 100 * avail["ga4_available_rows"] / avail["total_rows"]
avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.206382


### 3b. Five features, from the safe prev30 window (March 2–31)

Every feature below is aggregated only from dates **before** the 2026-03-31 decision point, so
each is knowable at the decision moment by construction. One line each:

1. **`imp_prev30`** — sum of `gsc_impressions`, Mar 2–31. Knowable at the decision moment
   because every date it sums is already in the past relative to 2026-03-31.
2. **`clk_prev30`** — sum of `gsc_clicks`, same window. Same reasoning.
3. **`pos_prev30`** — average of `gsc_avg_position`, same window. Same reasoning.
4. **`active_days_prev30`** — count of days in the window with `gsc_impressions > 0`. A
   consistency signal (steady vs. bursty visibility); same window, same reasoning.
5. **`zero_click_days_prev30`** — count of days in the window with `gsc_clicks = 0` despite
   impressions existing. An "attention gap" signal; same window, same reasoning.


In [31]:
features = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                   AS imp_prev30,
        SUM(gsc_clicks)                                         AS clk_prev30,
        AVG(gsc_avg_position)                                   AS pos_prev30,
        COUNT(*) FILTER (WHERE gsc_impressions > 0)             AS active_days_prev30,
        COUNT(*) FILTER (WHERE gsc_impressions > 0 AND gsc_clicks = 0) AS zero_click_days_prev30
    FROM {TABLES['fact_daily_march']}
    WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev30 >= 100
""").df()

print(f"{len(features):,} content items with enough March history")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

100,849 content items with enough March history


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,active_days_prev30,zero_click_days_prev30
0,client_62f4a7e64f5e0096,content_5d562ebeda84bdc7,773.0,0.0,11.463337,30,30
1,client_62f4a7e64f5e0096,content_d6f555d072e070be,25534.0,72.0,4.128734,30,5
2,client_62f4a7e64f5e0096,content_dabbfdf80d8773f0,1448.0,9.0,4.668850,30,23
3,client_62f4a7e64f5e0096,content_a761d62e362213d3,5595.0,24.0,4.035987,30,16
4,client_62f4a7e64f5e0096,content_eb6738715ef9bac1,412.0,2.0,9.107989,30,28


### 3c. The trap: add one label-derived column on purpose

First build the label (April outcome), join it to the honest features, and get a real baseline
score. Then deliberately add a column that's basically the label in disguise, watch the score
jump toward perfect, and remove it again.


In [32]:
# Build the label from a genuinely future window: April 1-30, never touched by the features above.
label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
    FROM {TABLES['fact_daily_april']}
    GROUP BY 1, 2
""").df()

data = features.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_apr"] < 0.8 * data["imp_prev30"]).astype(int)
print(f"joined rows with both March features and an April outcome: {len(data):,}")
print(f"decline rate: {data['is_declining'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined rows with both March features and an April outcome: 100,849
decline rate: 0.498


In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ["imp_prev30", "clk_prev30", "pos_prev30", "active_days_prev30", "zero_click_days_prev30"]

def quick_auc(cols):
    d = data.dropna(subset=cols + ["is_declining"])
    X, y = d[cols], d["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_cols)
print(f"HONEST auc (5 features, no leak): {honest_auc:.3f}")


HONEST auc (5 features, no leak): 0.633


In [34]:
# Now spring the trap: add ONE label-derived column on purpose.
# pct_change_mar_to_apr is computed FROM imp_apr -- the exact quantity the label is defined
# from. This is leakage dressed up as a feature.
data["pct_change_mar_to_apr"] = (data["imp_apr"] - data["imp_prev30"]) / data["imp_prev30"]

leaked_cols = honest_cols + ["pct_change_mar_to_apr"]
leaked_auc = quick_auc(leaked_cols)
print(f"LEAKED auc (honest features + 1 label-derived column): {leaked_auc:.3f}")
print(f"jump: {leaked_auc - honest_auc:+.3f}")


LEAKED auc (honest features + 1 label-derived column): 1.000
jump: +0.367


In [35]:
# Delete the leaked column and keep the honest number.
data = data.drop(columns=["pct_change_mar_to_apr"])
final_auc = quick_auc(honest_cols)

print(f"FINAL, honest auc (leak removed): {final_auc:.3f}")
assert abs(final_auc - honest_auc) < 0.05, "should land back near the original honest score"
print("This is the number that goes in any claim about this lane -- not the leaked one.")


FINAL, honest auc (leak removed): 0.633
This is the number that goes in any claim about this lane -- not the leaked one.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: `fact_content_query_90d`'s window is not re-cut per decision point.** It's
described as a fixed 90-day snapshot, not a table you can safely re-window like the daily fact
table. For a March 2026 decision point, its 90-day span likely runs well into or past April —
exactly the label window this notebook uses. Query-mix signals (`visible_queries`, `rare_share`,
`top_query_share`) are genuinely useful for this lane, but only once I've confirmed this table's
actual date coverage; used blindly, they'd be a second, quieter version of the trap in Section
3c. The query below is the check I'd run before ever adding those columns as features.

**Other real limits of this slice:**
- **Unbalanced panel.** History depth differs per client (`dim_clients.gsc_data_start` varies)
  — some clients may not have March 2026 data at all, so this slice isn't every client, it's
  every client with enough history by that point.
- **GA4 rows before `ga4_data_available`** are zero-filled, not truly zero — Query 3 above
  measures exactly how much of March is affected.
- **This is one month, one decision point.** A single March→April pair says nothing about
  whether the pattern holds in a different month; the capstone will need this repeated across
  more than one window before trusting it.


In [36]:
# Check: does fact_content_query_90d's snapshot actually postdate the March/April window
# used above? If dim_content or the query table exposes any date/recency column, compare it
# here. If no date column exists on the table itself, compare its row-level activity against
# a table that does have dates, or treat the window as unverified and keep it excluded.
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']} LIMIT 1").df()[["column_name", "column_type"]])

# If a date/recency column shows up above, replace this cell with the real MIN/MAX check
# against the March/April window before ever using this table as a feature source.


                      column_name column_type
0                  client_hash_id     VARCHAR
1                 content_hash_id     VARCHAR
2                   query_hash_id     VARCHAR
3                query_char_count      BIGINT
4               query_token_count      BIGINT
5                    window_start        DATE
6                      window_end        DATE
7                 impressions_90d      BIGINT
8                      clicks_90d      BIGINT
9              impressions_last30      BIGINT
10                  clicks_last30      BIGINT
11             impressions_prev30      BIGINT
12                  clicks_prev30      BIGINT
13               avg_position_90d      DOUBLE
14            avg_position_last30      DOUBLE
15            avg_position_prev30      DOUBLE
16  content_total_impressions_90d      BIGINT
17    content_visible_query_count      BIGINT
18               rare_query_count      BIGINT
19         rare_impressions_share      DOUBLE
20   anonymized_impressions_share 

In [37]:
# Real check: does fact_content_query_90d's 90-day window actually overlap
# the April label window (2026-04-01 -> 2026-04-30), or the March feature
# window (2026-03-02 -> 2026-03-31)? This replaces the "unverified" guess
# in the limitation above with a checked fact.
window_check = con.sql(f"""
    SELECT
        MIN(window_start) AS earliest_start,
        MAX(window_start) AS latest_start,
        MIN(window_end)   AS earliest_end,
        MAX(window_end)   AS latest_end,
        COUNT(*) FILTER (WHERE window_end > DATE '2026-03-31') AS rows_extending_past_march,
        COUNT(*) FILTER (WHERE window_start < DATE '2026-04-01' AND window_end >= DATE '2026-04-01')
            AS rows_overlapping_april,
        COUNT(*) AS total_rows
    FROM {TABLES['fact_query_90d']}
""").df()
window_check["pct_overlapping_april"] = 100 * window_check["rows_overlapping_april"] / window_check["total_rows"]
window_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_start,latest_start,earliest_end,latest_end,rows_extending_past_march,rows_overlapping_april,total_rows,pct_overlapping_april
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30,2414248,0,2414248,0.0
